In [1]:
   import pandas as pd, numpy as np
   import plotly.express as px, plotly.graph_objects as go

   WEB = r'C:\Users\ahmed\OneDrive\Desktop\DATA ENGGG'
   df_housing_monthly = pd.read_parquet(WEB + r'\housing_monthly.parquet')
   df_crashes = pd.read_parquet(WEB + r'\crashes.parquet')

   print(df_housing_monthly.shape, df_crashes.shape)
   print(df_housing_monthly.columns.tolist())
   print(df_crashes.columns.tolist())

(73104, 7) (11953, 17)
['cd_id', 'borough', 'year', 'month', 'major_category', 'type', 'problems']
['open_data_crash_id', 'crash_date', 'year', 'month', 'weekday', 'crash_hour', 'borough', 'cd_id', 'zip_code', 'on_street', 'total_severe_injuries', 'total_non_severe_injuries', 'total_fatalities', 'total_victims', 'latitude', 'longitude', 'in_integration']


   ### Q4. Do housing problems and serious crashes peak in the same months?
   Housing complaints follow the heating season, while crashes may follow traffic, weather and daylight. If the two peak in different months, then seasonal timing gives no shared signal. Comparing them shows whether inspections and street-safety work should be scheduled together or separately.

In [2]:
   from plotly.subplots import make_subplots

   MONTH_NAMES = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

   def build_q4(df_housing_monthly, df_crashes):
       """Q4: Do housing problems and serious crashes peak in the same months?"""
       months = range(1, 13)
       # divide by the number of years actually present, so a one-year filter stays correct
       h_years = max(df_housing_monthly['year'].nunique(), 1)
       c_years = max(df_crashes['year'].nunique(), 1)

       housing = (df_housing_monthly.groupby('month')['problems'].sum()
                  .reindex(months, fill_value=0) / h_years)
       heat = (df_housing_monthly[df_housing_monthly['major_category'] == 'HEAT/HOT WATER']
               .groupby('month')['problems'].sum()
               .reindex(months, fill_value=0) / h_years)
       crashes = (df_crashes.groupby('month').size()
                  .reindex(months, fill_value=0) / c_years)

       fig = make_subplots(specs=[[{"secondary_y": True}]])
       fig.add_trace(go.Scatter(x=MONTH_NAMES, y=housing.values, name='All housing problems (left axis)',
                                mode='lines+markers', line=dict(width=3),
                                hovertemplate='%{x}: %{y:,.0f} problems/yr<extra>All housing</extra>'),
                     secondary_y=False)
       fig.add_trace(go.Scatter(x=MONTH_NAMES, y=heat.values, name='Heat/hot-water problems (left axis)',
                                mode='lines', line=dict(dash='dot'),
                                hovertemplate='%{x}: %{y:,.0f} problems/yr<extra>Heat/hot water</extra>'),
                     secondary_y=False)
       fig.add_trace(go.Scatter(x=MONTH_NAMES, y=crashes.values, name='Serious crashes (right axis)',
                                mode='lines+markers', line=dict(width=3),
                                hovertemplate='%{x}: %{y:,.0f} crashes/yr<extra>Serious crashes</extra>'),
                     secondary_y=True)

       fig.update_layout(title='Do housing problems and serious crashes peak in the same months?',
                         xaxis_title='Month', hovermode='x unified',
                         legend=dict(orientation='h', y=-0.25))
       fig.update_yaxes(title_text='Housing problems (avg per year)', secondary_y=False, rangemode='tozero')
       fig.update_yaxes(title_text='Serious crashes (avg per year)', secondary_y=True, rangemode='tozero')
       return fig

   build_q4(df_housing_monthly, df_crashes).show()

   **Answer.** Housing problems peak in December, at about 75,054 per year, while serious crashes peak in July, at about 307 per year, so the two peaks do not fall in the same month. Heat and hot-water problems are highest in December (about 28,741 per year) and lowest in August (about 1,975), which matches the October–May heating season. The two monthly profiles have a correlation of r = -0.55, so months with more housing problems tend to be months with fewer serious crashes, a moderate negative association. Limitation: this compares only 12 monthly averages from 4 years, so one unusual month can shift the pattern, and monthly totals say nothing about whether the same districts are affected in both domains.

### Q8. Which housing problem categories are most often emergencies, and does that differ by borough?
Emergency problems (for example no heat or a gas leak) need a faster response than non-emergency ones. Knowing which categories are emergencies most often shows where inspectors should be ready to respond quickly. Comparing boroughs shows whether the mix of urgency differs across the city.

In [5]:
def build_q8(df_housing_monthly):
    """Q8: Which housing problem categories are most often emergencies?"""
    d = (df_housing_monthly.groupby(['major_category', 'type'], as_index=False)['problems'].sum())
    d['total'] = d.groupby('major_category')['problems'].transform('sum')
    d['pct'] = d['problems'] / d['total'] * 100

    # "emergency" = any type containing EMERGENCY but not NON (e.g. EMERGENCY, IMMEDIATE EMERGENCY)
    is_emerg = d['type'].str.upper().str.contains('EMERGENCY') & ~d['type'].str.upper().str.contains('NON')
    d['is_emergency'] = is_emerg
    order = (d[d['is_emergency']].groupby('major_category')['pct'].sum()
             .reindex(d['major_category'].unique(), fill_value=0)
             .sort_values())
    cats = order.index.tolist()

    types = (d[['type', 'is_emergency']].drop_duplicates()
             .sort_values(['is_emergency', 'type'], ascending=[False, True])['type'].tolist())
    palette = px.colors.qualitative.Set2

    fig = go.Figure()
    for i, t in enumerate(types):
        s = d[d['type'] == t].set_index('major_category').reindex(cats)
        fig.add_trace(go.Bar(
            y=cats, x=s['pct'].fillna(0), name=t, orientation='h',
            marker_color=palette[i % len(palette)],
            customdata=np.c_[s['problems'].fillna(0), s['total'].fillna(0)],
            hovertemplate=('%{y}<br>' + t + ': %{x:.1f}%'
                           '<br>%{customdata[0]:,.0f} of %{customdata[1]:,.0f} problems<extra></extra>')))
    fig.update_layout(barmode='stack',
                      title='Which housing problem categories are most often emergencies?',
                      xaxis_title='Share of problems in category (%)', yaxis_title='Housing problem category',
                      xaxis_range=[0, 100], legend_title_text='Urgency type',
                      height=max(450, 28 * len(cats) + 200))
    return fig

print(df_housing_monthly['type'].unique())
print(df_housing_monthly['borough'].unique())
build_q8(df_housing_monthly).show()
build_q8(df_housing_monthly[df_housing_monthly['borough'] == 'BRONX']).show()

<ArrowStringArray>
['EMERGENCY', 'NON EMERGENCY', 'IMMEDIATE EMERGENCY']
Length: 3, dtype: str
<ArrowStringArray>
['MANHATTAN', 'BRONX', 'BROOKLYN', 'QUEENS', 'STATEN ISLAND']
Length: 5, dtype: str


**Answer.** Heat/hot water (608,708 problems) and line of travel (6,931) are classified as emergencies 100% of the time, followed by safety (98.7% of 81,540 problems), outside building (87.5%) and door/window (78.3%). Elevator problems are never classified as emergencies (0.0% of 7,335), and appliance (11.8%) and general (16.6%) problems are rarely emergencies. Across all categories, 58.3% of housing problems are emergencies. In the Bronx the highest category is line of travel at 100% and the lowest is elevator at 0.0%, the same extremes as the citywide picture, although this check only covers the top and bottom categories, not the full ranking. Limitation: the emergency label is assigned when the complaint is classified and reflects how the problem was reported, not necessarily how serious it was on inspection. Some categories have few problems (line of travel and elevator have about 7,000 each), so their percentages rest on much smaller counts than heat/hot water.